In [1]:
## Initialize Hyperparameters and import libraries

import numpy as np
import torch

from model import *
from utilities import *
from loss_ftn import *

In [2]:
PATH = "D:\\01_Datasets\\DIRTL_Manuscript\\"

data_dims = np.shape(np.load(PATH + "true_results_multiwl\\result_Hy_"+f'{0:08d}'+'k'+f"{0.01:.4f}"+".npy")[0])

In [3]:
n_train = 4500
n_test = 1000

batch_size = 250
epochs = 100

layer_num = 6

In [4]:
train_loader ,test_loader = data_loader(n_train, n_test, batch_size, wl_list, data_dims, 0.01, dz)

Loading Test Data: 100%|██████████| 1000/1000 [01:12<00:00, 13.89it/s]


In [5]:
model = FNOModel2d(modes=16, width=32, blocks=layer_num).cuda()

In [6]:
# # 6 Layers
lr_top = 0.001
step_size = 30
gamma = 0.5

# 10 Layers
# lr_top = 0.0005
# step_size = 30
# gamma = 0.5

In [7]:
name = (n_train/1000)
SAVE_model = f"DIRTL_{name:.2f}k_pretrain_{layer_num}FL.pth"
SAVE_lc    = f"DIRTL_{name:.2f}k_pretrain_{layer_num}FL_LC.npz"

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr_top)

Train_rel_L1_arr = []
Train_rel_L2_arr = []
Test_rel_L1_arr = []
Test_rel_L2_arr = []

# Define StepLR scheduler
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=step_size,    # number of epochs before decay
    gamma=gamma             # decay factor, e.g., 0.1 reduces LR by 10x
)

loss = nn.MSELoss()

# gc.collect(k)
torch.cuda.empty_cache()

total_time = 0

for ep in range(epochs):
    t1 = default_timer()
    model.train()
    Train_mse = 0

    for input_shape, result in train_loader:
        input_shape, result = input_shape.cuda(), result.cuda()
        optimizer.zero_grad()
        
        out = model((input_shape))

        Train_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))

        Train_mse_temp.backward()
        
        optimizer.step()
        
        Train_mse += Train_mse_temp.detach() * batch_size

    scheduler.step()

    model.eval()
    Test_mse = 0.0
    with torch.no_grad():
        for input_shape, result in test_loader:
            input_shape, result = input_shape.cuda(), result.cuda()

            out = model((input_shape))
            Test_mse_temp = loss(out.reshape(batch_size, -1), result.reshape(batch_size, -1))
            Test_mse += Test_mse_temp.detach() * batch_size

    Train_mse /= len(train_loader.dataset)
    Test_mse /= len(test_loader.dataset)

    Train_rmse = np.sqrt(Train_mse.item())
    Test_rmse = np.sqrt(Test_mse.item())

    if SAVE_lc:
        with torch.no_grad():
            model.eval()
            _, _, rel_L1, rel_L2, _ = rel_err(model, train_loader)
            Train_rel_L1_arr.append(np.mean(rel_L1))
            Train_rel_L2_arr.append(np.mean(rel_L2))

            _, _, rel_L1, rel_L2, _ = rel_err(model, test_loader)
            Test_rel_L1_arr.append(np.mean(rel_L1))
            Test_rel_L2_arr.append(np.mean(rel_L2))
        
    t2 = default_timer()
    total_time += t2 - t1

    print(f"Epoch {ep+1}, Time: {t2-t1:.2f}s, Train RMSE: {Train_rmse:.4f}, Test RMSE: {Test_rmse:.4f}")

print(f"total time: {total_time:.2f}")

if SAVE_model:
    torch.save(model.state_dict(), SAVE_model)

if SAVE_lc:
    np.savez(SAVE_lc,
        Train_rel_L1=Train_rel_L1_arr,
        Train_rel_L2=Train_rel_L2_arr,
        Test_rel_L1=Test_rel_L1_arr,
        Test_rel_L2=Test_rel_L2_arr)

Epoch 1, Time: 48.88s, Train RMSE: 0.3322, Test RMSE: 0.1890
Epoch 2, Time: 46.25s, Train RMSE: 0.1773, Test RMSE: 0.1639
Epoch 3, Time: 45.92s, Train RMSE: 0.1497, Test RMSE: 0.1553
Epoch 4, Time: 45.90s, Train RMSE: 0.1192, Test RMSE: 0.1051
Epoch 5, Time: 45.86s, Train RMSE: 0.1056, Test RMSE: 0.1006
Epoch 6, Time: 46.05s, Train RMSE: 0.0918, Test RMSE: 0.0884
Epoch 7, Time: 45.87s, Train RMSE: 0.0801, Test RMSE: 0.0751
Epoch 8, Time: 45.79s, Train RMSE: 0.0789, Test RMSE: 0.0682
Epoch 9, Time: 45.96s, Train RMSE: 0.0676, Test RMSE: 0.0759
Epoch 10, Time: 45.78s, Train RMSE: 0.0615, Test RMSE: 0.0516
Epoch 11, Time: 45.90s, Train RMSE: 0.0597, Test RMSE: 0.0645
Epoch 12, Time: 46.47s, Train RMSE: 0.0654, Test RMSE: 0.0723
Epoch 13, Time: 46.64s, Train RMSE: 0.0732, Test RMSE: 0.0514
Epoch 14, Time: 46.90s, Train RMSE: 0.0611, Test RMSE: 0.0468
Epoch 15, Time: 46.92s, Train RMSE: 0.0555, Test RMSE: 0.0432
Epoch 16, Time: 46.22s, Train RMSE: 0.0468, Test RMSE: 0.0429
Epoch 17, Time: 4